# Thermocycler quickstart

This notebook shows the basic thermocycler workflow in PyLabRobot using the `ThermocyclerChatterboxBackend`. The chatterbox backend prints actions instead of controlling real hardware, which makes it useful for learning and testing examples.


## Create a thermocycler

A thermocycler combines the high-level `Thermocycler` resource with a backend. For this quickstart, we use `ThermocyclerChatterboxBackend` so the notebook can run without physical hardware.


In [ ]:
from pylabrobot.resources import Coordinate
from pylabrobot.thermocycling import Thermocycler, ThermocyclerChatterboxBackend
from pylabrobot.thermocycling.standard import Protocol, Stage, Step

tc = Thermocycler(
  name="tc",
  size_x=1,
  size_y=1,
  size_z=1,
  backend=ThermocyclerChatterboxBackend(),
  child_location=Coordinate.zero(),
)


## Lid control

Use `open_lid` and `close_lid` to control the thermocycler lid.


In [ ]:
await tc.open_lid()
await tc.close_lid()


## Temperature control

Set the block and lid temperatures. Temperatures are passed as lists because some thermocyclers support multiple temperature zones.


In [ ]:
await tc.set_block_temperature([95.0])
await tc.set_lid_temperature([105.0])


## Query status

You can query temperatures, lid state, block status, lid status, and profile progress.


In [ ]:
block_temperature = await tc.get_block_current_temperature()
lid_temperature = await tc.get_lid_current_temperature()
lid_open = await tc.get_lid_open()
block_status = await tc.get_block_status()
lid_status = await tc.get_lid_status()

block_temperature, lid_temperature, lid_open, block_status, lid_status


## Run a custom protocol

A protocol contains one or more stages. Each stage contains one or more steps and a repeat count.


In [ ]:
protocol = Protocol(
  stages=[
    Stage(
      steps=[
        Step(temperature=[95.0], hold_seconds=10),
        Step(temperature=[55.0], hold_seconds=20),
      ],
      repeats=1,
    )
  ]
)

await tc.run_protocol(protocol, block_max_volume=25.0)
await tc.wait_for_profile_completion(poll_interval=0.01)


## Run a PCR profile

`run_pcr_profile` builds a standard PCR-style protocol from denaturation, annealing, extension, and optional storage parameters.


In [ ]:
await tc.run_pcr_profile(
  denaturation_temp=[98.0],
  denaturation_time=15.0,
  annealing_temp=[60.0],
  annealing_time=15.0,
  extension_temp=[72.0],
  extension_time=20.0,
  num_cycles=2,
  block_max_volume=25.0,
  lid_temperature=[105.0],
  storage_temp=[4.0],
  storage_time=1.0,
)
await tc.wait_for_profile_completion(poll_interval=0.01)


## Shut down heaters

Deactivate the block and lid heaters when they are no longer needed.


In [ ]:
await tc.deactivate_block()
await tc.deactivate_lid()
